# OffScript — Phase 4: Batter Data Collection

This notebook builds batter vulnerability profiles by extracting batter
performance against each pitch type directly from the existing pitcher
dataset. This avoids a separate large Statcast pull — the pitcher data
already contains the batter-side outcome information needed.

**Goals:**
- Identify the batters most frequently represented in the pitcher dataset
- Compute per-batter performance metrics against each pitch type
- Resolve batter MLBAM IDs to human-readable names
- Score each batter's vulnerability to each pitch type
- Persist batter profiles for use in notebook 08

**Input:** `data/processed/pitcher_data_clean.parquet`  
**Output:** `data/processed/batter_vs_pitch.parquet`, `data/processed/batter_vulnerability.parquet`

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")

import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pybaseball import statcast_batter, playerid_lookup, playerid_reverse_lookup, cache
from pitch_analysis import load_clean_data

# Prevent re-downloading Statcast data on every run
cache.enable()

print("Imports successful")

## 2. Load Pitcher Dataset

Rather than pulling a separate batter dataset, batter performance is
derived directly from the existing pitcher data. Each row in the pitcher
dataset represents a single pitch and already contains the batter MLBAM ID,
handedness, pitch type faced, and outcome — everything needed to build
batter vulnerability profiles.

In [ ]:
pitcher_data = load_clean_data()

print(f"Dataset loaded: {len(pitcher_data):,} pitches")
print(f"\nBatter-related columns available:")
batter_cols = [
    col for col in pitcher_data.columns
    if any(term in col.lower() for term in ("batter", "bat", "stand"))
]
print(batter_cols)

print("\n=== Sample Batter Records ===")
print(pitcher_data[["batter", "stand", "pitch_type", "events", "description"]].head(10))

## 3. Identify Batter Pool

Batters are ranked by frequency of appearance in the pitcher dataset.
The top 50 most frequently faced batters are selected — these have the
largest sample sizes against the pitcher pool, making their vulnerability
scores the most statistically reliable.

In [ ]:
TOP_BATTERS = 50
MIN_PITCHES_SEEN = 20

batter_counts = pitcher_data["batter"].value_counts()

print("=== Batter Exposure in Dataset ===")
print(f"Unique batters faced: {len(batter_counts):,}")
print(f"\nTop 30 most frequently faced:")
print(batter_counts.head(30))

top_batter_ids = batter_counts.head(TOP_BATTERS).index.tolist()
print(f"\nSelected top {TOP_BATTERS} batters for analysis")

## 4. Extract Batter Performance by Pitch Type

For each batter-handedness-pitch type combination, the following metrics
are aggregated from the pitcher dataset:

| Metric | Definition |
|---|---|
| `pitches_seen` | Total pitches of this type faced |
| `swinging_strikes` | Pitches where batter swung and missed |
| `called_strikes` | Pitches called strike without a swing |
| `balls_in_play` | Pitches put into play (any outcome) |
| `hits` | Singles, doubles, triples, home runs |
| `strikeouts` | At-bats ending in strikeout |
| `whiff_rate` | `swinging_strikes / pitches_seen` |
| `hit_rate` | `hits / pitches_seen` |

Batter-pitch combinations with fewer than `MIN_PITCHES_SEEN` pitches
are excluded — small samples produce unreliable rate statistics.

In [ ]:
batter_vs_pitch = pitcher_data.groupby(
    ["batter", "stand", "pitch_type"]
).agg(
    pitches_seen=("pitch_type", "count"),
    swinging_strikes=("description", lambda x: (x == "swinging_strike").sum()),
    called_strikes=("description", lambda x: (x == "called_strike").sum()),
    balls_in_play=("events", lambda x: x.isin([
        "single", "double", "triple", "home_run",
        "field_out", "grounded_into_double_play", "force_out", "sac_fly",
    ]).sum()),
    hits=("events", lambda x: x.isin(["single", "double", "triple", "home_run"]).sum()),
    strikeouts=("events", lambda x: (x == "strikeout").sum()),
    walks=("events", lambda x: (x == "walk").sum()),
    home_runs=("events", lambda x: (x == "home_run").sum()),
).reset_index()

# Compute rate statistics
batter_vs_pitch["whiff_rate"] = (
    batter_vs_pitch["swinging_strikes"] / batter_vs_pitch["pitches_seen"]
).round(3)
batter_vs_pitch["contact_rate"] = (
    batter_vs_pitch["balls_in_play"] / batter_vs_pitch["pitches_seen"]
).round(3)
batter_vs_pitch["hit_rate"] = (
    batter_vs_pitch["hits"] / batter_vs_pitch["pitches_seen"].clip(lower=1)
).round(3)

# Require minimum sample size for reliable rate estimates
batter_vs_pitch = batter_vs_pitch[batter_vs_pitch["pitches_seen"] >= MIN_PITCHES_SEEN]

print("=== Batter vs Pitch Type Performance ===")
print(f"Batter-pitch combinations (≥{MIN_PITCHES_SEEN} pitches): {len(batter_vs_pitch):,}")
print(f"Unique batters:                                          {batter_vs_pitch['batter'].nunique():,}")
print(f"\nSample records:")
print(batter_vs_pitch.head(10).to_string(index=False))

## 5. Resolve Batter Names

Statcast stores batters as MLBAM integer IDs. `playerid_reverse_lookup`
maps those IDs back to human-readable names. If the lookup fails the
notebook continues with numeric IDs so downstream cells are not blocked.

In [ ]:
unique_batter_ids = batter_vs_pitch["batter"].unique().tolist()
print(f"Resolving names for {len(unique_batter_ids):,} batters...")

try:
    batter_names = playerid_reverse_lookup(unique_batter_ids, key_type="mlbam")
    batter_names = batter_names[["key_mlbam", "name_first", "name_last"]].copy()
    batter_names["batter_name"] = batter_names["name_first"] + " " + batter_names["name_last"]
    batter_names = batter_names.rename(columns={"key_mlbam": "batter"})

    batter_vs_pitch = batter_vs_pitch.merge(
        batter_names[["batter", "batter_name"]], on="batter", how="left"
    )

    print(f"Names resolved for {batter_names.shape[0]:,} batters")
    print("\nSample with names:")
    print(
        batter_vs_pitch[["batter_name", "stand", "pitch_type", "pitches_seen", "whiff_rate", "hit_rate"]]
        .head(10)
        .to_string(index=False)
    )

except Exception as e:
    print(f"Name lookup failed: {e}")
    print("Continuing with numeric batter IDs.")
    batter_vs_pitch["batter_name"] = batter_vs_pitch["batter"].astype(str)

## 6. Build Batter Vulnerability Profiles

A vulnerability score is computed for each batter-pitch type combination.
The score combines two components:

- **Whiff rate (60% weight):** A high whiff rate indicates the batter
  struggles to make contact against this pitch type
- **Inverse hit rate (40% weight):** A low hit rate indicates the batter
  makes little productive contact when they do swing

The score is scaled to 0–100 for interpretability. Higher scores indicate
greater batter vulnerability to a given pitch type.

In [ ]:
# Whiff rate weighted higher than inverse hit rate — missing the ball
# entirely is a stronger vulnerability signal than weak contact
WHIFF_WEIGHT   = 0.6
CONTACT_WEIGHT = 0.4

vulnerability_records = []

for _, row in batter_vs_pitch.iterrows():
    score = (
        row["whiff_rate"] * WHIFF_WEIGHT
        + (1 - row["hit_rate"]) * CONTACT_WEIGHT
    ) * 100

    vulnerability_records.append({
        "batter":              row["batter"],
        "batter_name":         row["batter_name"],
        "stand":               row["stand"],
        "pitch_type":          row["pitch_type"],
        "pitches_seen":        row["pitches_seen"],
        "whiff_rate":          row["whiff_rate"],
        "hit_rate":            row["hit_rate"],
        "vulnerability_score": round(score, 2),
    })

vulnerability_df = pd.DataFrame(vulnerability_records)

print("=== Vulnerability Score Distribution by Pitch Type ===")
print(vulnerability_df.groupby("pitch_type")["vulnerability_score"].describe().round(2))

## 7. Pivot to Per-Batter Profile Matrix

In [ ]:
batter_pivot = batter_vs_pitch.pivot_table(
    index=["batter", "batter_name", "stand"],
    columns="pitch_type",
    values=["whiff_rate", "hit_rate", "pitches_seen"],
).reset_index()

# Flatten MultiIndex columns to single-level strings
batter_pivot.columns = [
    "_".join(col).strip("_") if col[1] else col[0]
    for col in batter_pivot.columns
]

print(f"Batters profiled: {len(batter_pivot):,}")
print(f"Profile columns:  {list(batter_pivot.columns)}")

## 8. Save Batter Data

In [ ]:
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

outputs = {
    "batter_vs_pitch.parquet":    batter_vs_pitch,
    "batter_vulnerability.parquet": vulnerability_df,
}

for filename, df in outputs.items():
    path = f"{OUTPUT_DIR}/{filename}"
    df.to_parquet(path, index=False)
    print(f"Saved: {path} ({len(df):,} records)")